In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
import datetime

In [2]:
# ==========================================
# 1. Load & Merge Data (ترکیب جداول)
# ==========================================
print("Loading and Merging Data...")

train = pd.read_csv('train.csv', parse_dates=['Date'], low_memory=False)
store = pd.read_csv('store.csv')

# ادغام دو جدول بر اساس Store
df = pd.merge(train, store, on='Store', how='left')

# فقط فروشگاه‌های باز که فروش داشته‌اند (برای آموزش مدل)
# نکته: برای تست نهایی ممکن است نیاز باشد روزهای بسته را هم نگه دارید، اما برای آموزش مدل معمولا حذف می‌شوند
df = df[(df['Open'] == 1) & (df['Sales'] > 0)].copy()

# مرتب‌سازی زمانی (بسیار مهم برای ساخت ویژگی‌های Lag)
df = df.sort_values(['Store', 'Date']).reset_index(drop=True)

print(f"Merged Shape: {df.shape}")

Loading and Merging Data...
Merged Shape: (830918, 19)


In [3]:
# ==========================================
# 2. Temporal Features (ویژگی‌های زمانی جدید)
# ==========================================
print("Generating Basic Temporal Features...")

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['WeekOfYear'] = df['Date'].dt.isocalendar().week.astype(int)
df['IsWeekend'] = df['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)

# محاسبه روزهای گذشته از شروع رقابت (Competition)
# (تبدیل سال و ماه به یک تاریخ تقریبی)
df['CompetitionOpenSince'] = pd.to_datetime(dict(year=df.CompetitionOpenSinceYear.fillna(1900), 
                                                 month=df.CompetitionOpenSinceMonth.fillna(1), 
                                                 day=1))
df['CompetitionDaysOpen'] = (df['Date'] - df['CompetitionOpenSince']).dt.days
# اگر مقدار منفی شد (هنوز باز نشده) یا داده نداشت، صفر می‌کنیم
df['CompetitionDaysOpen'] = df['CompetitionDaysOpen'].apply(lambda x: x if x > 0 else 0)

Generating Basic Temporal Features...


In [4]:
# ==========================================
# 3. Lag Features & Rolling Statistics
# ==========================================
print("Generating Lag & Rolling Features (This may take time)...")

# تابع گروپ‌بای برای اعمال روی هر فروشگاه جداگانه
grouped = df.groupby('Store')['Sales']

# الف) Lag Features: فروش 1 روز قبل، 2 روز قبل و ...
# نکته: ما از shift استفاده می‌کنیم تا از "آینده" خبر نداشته باشیم
df['Sales_Lag1'] = grouped.shift(1)
df['Sales_Lag2'] = grouped.shift(2)
df['Sales_Lag7'] = grouped.shift(7) # فروش هفته گذشته همان روز

# ب) Rolling Features: میانگین متحرک 7 روز اخیر
# نکته مهم: ابتدا shift(1) می‌کنیم تا فروش "امروز" در میانگین "دیروز و قبلتر" نیاید
df['Sales_RollMean7'] = grouped.shift(1).rolling(window=7).mean()
df['Sales_RollStd7']  = grouped.shift(1).rolling(window=7).std()

# حذف ردیف‌هایی که به خاطر Lag مقدار NaN گرفتند (چند روز اول هر فروشگاه)
df.dropna(subset=['Sales_Lag1', 'Sales_Lag7', 'Sales_RollMean7'], inplace=True)

Generating Lag & Rolling Features (This may take time)...


In [5]:
# ==========================================
# 4. Fourier Terms (ویژگی‌های دوره‌ای)
# ==========================================
print("Generating Fourier Terms...")

# برای فصلی بودن هفتگی (Weekly Seasonality) - دوره 7 روزه
df['day_sin'] = np.sin(2 * np.pi * df['DayOfWeek'] / 7)
df['day_cos'] = np.cos(2 * np.pi * df['DayOfWeek'] / 7)

# برای فصلی بودن سالانه (Yearly Seasonality) - دوره 365 روزه
day_of_year = df['Date'].dt.dayofyear
df['year_sin'] = np.sin(2 * np.pi * day_of_year / 365)
df['year_cos'] = np.cos(2 * np.pi * day_of_year / 365)

Generating Fourier Terms...


In [6]:
# ==========================================
# 5. Missing Values & Encoding
# ==========================================
print("Handling Missing Values & Encoding...")

# پر کردن مقادیر گمشده ستون‌های خاص
df['CompetitionDistance'].fillna(df['CompetitionDistance'].median(), inplace=True)
df.fillna(0, inplace=True) # سایر مقادیر باقی‌مانده

# Encoding متغیرهای دسته‌ای (Categorical)
categorical_cols = ['StateHoliday', 'StoreType', 'Assortment']
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    # تبدیل به رشته برای اطمینان
    df[col] = df[col].astype(str)
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

Handling Missing Values & Encoding...


C:\Users\AHFB\AppData\Local\Temp\ipykernel_10812\1128958453.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['CompetitionDistance'].fillna(df['CompetitionDistance'].median(), inplace=True)


In [7]:
# ==========================================
# 6. Normalization (نرمال‌سازی)
# ==========================================
print("Normalizing Numerical Features...")

# انتخاب ستون‌های عددی که نیاز به اسکیل دارند (به جز هدف و ویژگی‌های قطعی)
num_cols = ['CompetitionDistance', 'CompetitionDaysOpen', 
            'Sales_RollMean7', 'Sales_RollStd7']

scaler = MinMaxScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

Normalizing Numerical Features...


In [8]:
# ==========================================
# 7. Time-Series Split (تفکیک داده با حفظ زمان)
# ==========================================
print("Splitting Data (Train/Val/Test)...")

# چون داده سری زمانی است، نباید Shuffle کنیم.
# استراتژی: 6 هفته آخر برای تست، 6 هفته قبل از آن برای اعتبارسنجی (Validation)

# پیدا کردن آخرین تاریخ در دیتاست
max_date = df['Date'].max()
test_cutoff = max_date - pd.Timedelta(days=6*7) # 6 هفته آخر
val_cutoff = test_cutoff - pd.Timedelta(days=6*7) # 6 هفته قبل‌تر

train_df = df[df['Date'] < val_cutoff]
val_df   = df[(df['Date'] >= val_cutoff) & (df['Date'] < test_cutoff)]
test_df  = df[df['Date'] >= test_cutoff]

# انتخاب فیچرها و تارگت
target = 'Sales'
ignore_cols = ['Date', 'Customers', 'Open', 'CompetitionOpenSince', 
               'CompetitionOpenSinceYear', 'CompetitionOpenSinceMonth']
features = [c for c in df.columns if c not in ignore_cols + [target]]

print("-" * 30)
print(f"Training Set:   {train_df.shape[0]} rows (Dates: {train_df.Date.min().date()} to {train_df.Date.max().date()})")
print(f"Validation Set: {val_df.shape[0]} rows   (Dates: {val_df.Date.min().date()} to {val_df.Date.max().date()})")
print(f"Test Set:       {test_df.shape[0]} rows   (Dates: {test_df.Date.min().date()} to {test_df.Date.max().date()})")
print("-" * 30)
print(f"Final Features List ({len(features)}): \n{features}")

# آماده‌سازی X و y نهایی
X_train, y_train = train_df[features], train_df[target]
X_val, y_val     = val_df[features], val_df[target]
X_test, y_test   = test_df[features], test_df[target]

print("\nPhase 2 Completed Successfully.")

Splitting Data (Train/Val/Test)...
------------------------------
Training Set:   745322 rows (Dates: 2013-01-08 to 2015-04-23)
Validation Set: 36376 rows   (Dates: 2015-04-24 to 2015-06-04)
Test Set:       41415 rows   (Dates: 2015-06-05 to 2015-07-17)
------------------------------
Final Features List (28): 
['Store', 'DayOfWeek', 'Promo', 'StateHoliday', 'SchoolHoliday', 'Id', 'StoreType', 'Assortment', 'CompetitionDistance', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval', 'Year', 'Month', 'Day', 'WeekOfYear', 'IsWeekend', 'CompetitionDaysOpen', 'Sales_Lag1', 'Sales_Lag2', 'Sales_Lag7', 'Sales_RollMean7', 'Sales_RollStd7', 'day_sin', 'day_cos', 'year_sin', 'year_cos']

Phase 2 Completed Successfully.
